In [3]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
# Asegúrate de que el archivo 'supply_chain_data.csv' esté en el mismo directorio que este script
file_path = 'supply_chain_data.csv'
df = pd.read_csv(file_path)

print("✅ Dataset cargado correctamente.")
print(f"Dimensiones originales: {df.shape[0]} filas, {df.shape[1]} columnas\n")

# 2. Definir el diccionario de mapeo para la estandarización de nomenclatura
column_mapping = {
    'Lead times': 'Supplier_Lead_Time_Days',
    'Lead time': 'Manufacturing_Lead_Time_Days',
    'Costs': 'Total_Operational_Costs',
    'Defect rates': 'Defect_Rate_Percentage'
}

# 3. Renombrar las columnas
# Usamos errors='raise' para asegurarnos de que si una columna no existe, el código nos avise
df.rename(columns=column_mapping, inplace=True)

print("✅ Columnas renombradas exitosamente.")

# 4. Asegurar que 'Defect_Rate_Percentage' esté en escala 0-100
# Lógica: Si el valor máximo es <= 1.0, asumimos que está en formato decimal (ej. 0.048 = 4.8%) y lo convertimos.
# Si el valor ya es > 1.0 (ej. 4.85), asumimos que ya está en formato porcentaje y solo lo limpiamos.
max_defect_rate = df['Defect_Rate_Percentage'].max()

if max_defect_rate <= 1.0:
    df['Defect_Rate_Percentage'] = df['Defect_Rate_Percentage'] * 100
    print(f"⚠️ Escala de defectos ajustada: se multiplicó por 100 (Máximo original era {max_defect_rate:.4f})")
else:
    print(f"ℹ️ Escala de defectos verificada: ya está en formato porcentaje (Máximo actual: {max_defect_rate:.2f}%)")

# Redondear a 2 decimales para mejorar la legibilidad y estandarización
df['Defect_Rate_Percentage'] = df['Defect_Rate_Percentage'].round(2)

# 5. Validación y limpieza de tipos de datos (Best Practice)
# Aseguramos que las nuevas columnas numéricas sean efectivamente numéricas
numeric_cols_to_check = [
    'Supplier_Lead_Time_Days', 
    'Manufacturing_Lead_Time_Days', 
    'Total_Operational_Costs', 
    'Defect_Rate_Percentage'
]

for col in numeric_cols_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 6. Verificación final de los cambios
print("\n" + "="*50)
print("📊 VERIFICACIÓN DE ESTANDARIZACIÓN")
print("="*50)

# Mostrar las columnas renombradas y sus estadísticas básicas
verification_df = df[numeric_cols_to_check].describe().round(2)
print("\nEstadísticas de las columnas estandarizadas:")
print(verification_df)

print("\nMuestra de las primeras 3 filas con los nuevos nombres:")
print(df[['Product type', 'SKU', 'Supplier_Lead_Time_Days', 'Manufacturing_Lead_Time_Days', 'Defect_Rate_Percentage', 'Total_Operational_Costs']].head(3))


df.to_csv('supply_chain_data.csv', index=False)


✅ Dataset cargado correctamente.
Dimensiones originales: 100 filas, 24 columnas

✅ Columnas renombradas exitosamente.
ℹ️ Escala de defectos verificada: ya está en formato porcentaje (Máximo actual: 4.94%)

📊 VERIFICACIÓN DE ESTANDARIZACIÓN

Estadísticas de las columnas estandarizadas:
       Supplier_Lead_Time_Days  Manufacturing_Lead_Time_Days  \
count                   100.00                        100.00   
mean                     15.96                         17.08   
std                       8.79                          8.85   
min                       1.00                          1.00   
25%                       8.00                         10.00   
50%                      17.00                         18.00   
75%                      24.00                         25.00   
max                      30.00                         30.00   

       Total_Operational_Costs  Defect_Rate_Percentage  
count                   100.00                  100.00  
mean                   

In [6]:
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURACIÓN: Ruta del único dataset que será leído y sobreescribido
# ==============================================================================
file_path = 'supply_chain_data.csv'

print(f"🔄 Cargando dataset desde: '{file_path}'...")
df = pd.read_csv(file_path)
registros_iniciales = len(df)

# ==============================================================================
# PASO 1: Estandarización de Nomenclatura (Data Dictionary)
# ==============================================================================
column_mapping = {
    'Lead times': 'Supplier_Lead_Time_Days',
    'Lead time': 'Manufacturing_Lead_Time_Days',
    'Costs': 'Total_Operational_Costs',
    'Defect rates': 'Defect_Rate_Percentage',
    # Renombrado preventivo debido a la inconsistencia del 100% detectada anteriormente
    'Revenue generated': 'Unverified_Revenue_or_Profit' 
}
df.rename(columns=column_mapping, inplace=True)

# ==============================================================================
# PASO 2: Corrección de Escala en Tasas de Defecto
# ==============================================================================
# Si los valores están en decimal (0-1), los convertimos a porcentaje (0-100)
if df['Defect_Rate_Percentage'].max() <= 1.0:
    df['Defect_Rate_Percentage'] = df['Defect_Rate_Percentage'] * 100

# Estandarizamos a 2 decimales para limpieza visual y de almacenamiento
df['Defect_Rate_Percentage'] = df['Defect_Rate_Percentage'].round(2)

# ==============================================================================
# PASO 3: Regla de Negocio 1 - Integridad de Ingresos
# ==============================================================================
# Calculamos el ingreso teórico esperado
df['Expected_Revenue'] = (df['Price'] * df['Number of products sold']).round(2)

# Calculamos la desviación absoluta
df['Revenue_Diff_Abs'] = np.abs(df['Unverified_Revenue_or_Profit'] - df['Expected_Revenue']).round(2)

# Bandera de alerta: True si la diferencia supera la tolerancia por redondeo (0.05)
df['Revenue_Mismatch_Flag'] = df['Revenue_Diff_Abs'] > 0.05

# ==============================================================================
# PASO 4: Regla de Negocio 2 - Lógica de Inventario (Backorders)
# ==============================================================================
# Si el stock es 0 pero se registraron ventas > 0, se marca como pedido pendiente (Backorder)
df['Is_Backorder'] = np.where(
    (df['Stock levels'] == 0) & (df['Number of products sold'] > 0), 
    True, 
    False
)

# ==============================================================================
# PASO 5: SOBREESCRITURA DEL DATASET (Single Source of Truth)
# ==============================================================================
# Guardamos los cambios directamente sobre el archivo original
df.to_csv(file_path, index=False)

print("✅ ¡Proceso completado exitosamente!")
print(f"💾 El archivo '{file_path}' ha sido sobreescribido con todas las mejoras.")
print(f"📊 Registros procesados: {registros_iniciales}")

print("\n" + "="*70)
print("📋 RESUMEN DE CAMBIOS APLICADOS AL DATASET:")
print("="*70)
print("1. Columnas renombradas: 'Lead times', 'Lead time', 'Costs', 'Defect rates', 'Revenue generated'")
print("2. 'Defect_Rate_Percentage' estandarizada a formato porcentaje con 2 decimales.")
print("3. Nuevas columnas de auditoría añadidas:")
print("   - 'Expected_Revenue' (Cálculo: Price * Number of products sold)")
print("   - 'Revenue_Diff_Abs' (Desviación absoluta)")
print("   - 'Revenue_Mismatch_Flag' (Alerta de inconsistencia)")
print("   - 'Is_Backorder' (Alerta de venta sin stock físico)")

# Verificación rápida en consola
print("\n🔍 Muestra de las nuevas columnas de control (Primeras 3 filas):")
cols_verificacion = [
    'SKU', 'Unverified_Revenue_or_Profit', 'Expected_Revenue', 
    'Revenue_Mismatch_Flag', 'Is_Backorder'
]
print(df[cols_verificacion].head(3).to_string(index=False))

🔄 Cargando dataset desde: 'supply_chain_data.csv'...
✅ ¡Proceso completado exitosamente!
💾 El archivo 'supply_chain_data.csv' ha sido sobreescribido con todas las mejoras.
📊 Registros procesados: 100

📋 RESUMEN DE CAMBIOS APLICADOS AL DATASET:
1. Columnas renombradas: 'Lead times', 'Lead time', 'Costs', 'Defect rates', 'Revenue generated'
2. 'Defect_Rate_Percentage' estandarizada a formato porcentaje con 2 decimales.
3. Nuevas columnas de auditoría añadidas:
   - 'Expected_Revenue' (Cálculo: Price * Number of products sold)
   - 'Revenue_Diff_Abs' (Desviación absoluta)
   - 'Revenue_Mismatch_Flag' (Alerta de inconsistencia)
   - 'Is_Backorder' (Alerta de venta sin stock físico)

🔍 Muestra de las nuevas columnas de control (Primeras 3 filas):
 SKU  Unverified_Revenue_or_Profit  Expected_Revenue  Revenue_Mismatch_Flag  Is_Backorder
SKU0                   8661.996792          55986.02                   True         False
SKU1                   7460.900065          10924.83                

In [1]:
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURACIÓN: Ruta del único dataset (Single Source of Truth)
# ==============================================================================
file_path = 'supply_chain_data.csv'

print(f"🔄 Cargando dataset desde: '{file_path}'...")
df = pd.read_csv(file_path)

# ==============================================================================
# PASO 0: Verificación de compatibilidad con transformaciones previas
# ==============================================================================
# El código es robusto: funciona tanto si las columnas ya fueron renombradas
# en pasos anteriores como si el dataset está en su estado original.

# Si la columna ya fue renombrada, usamos el nombre nuevo; si no, el original.
defect_col = 'Defect_Rate_Percentage' if 'Defect_Rate_Percentage' in df.columns else 'Defect rates'
inspection_col = 'Inspection results'

print(f"✅ Columna de defectos detectada: '{defect_col}'")
print(f"   Rango actual: {df[defect_col].min():.2f} - {df[defect_col].max():.2f}")

# ==============================================================================
# PASO 1: Guardar el estado original para auditoría (trazabilidad)
# ==============================================================================
# Creamos una columna que preserve la inspección original antes de recalcular.
# Esto es una buena práctica de gobernanza: nunca borres el dato fuente sin rastro.
df['Inspection_Original'] = df[inspection_col]

# ==============================================================================
# PASO 2: Recalcular 'Inspection results' con umbrales objetivos
# ==============================================================================
# Reglas de negocio definidas:
#   <= 1.0%  → 'Pass'    (Calidad aceptable, producto listo para despacho)
#   > 1.0% y <= 5.0% → 'Pending' (Requiere revisión, calidad marginal)
#   > 5.0%   → 'Fail'    (Calidad inaceptable, rechazar lote)

def clasificar_calidad(defect_rate):
    """
    Clasifica el resultado de inspección basado en la tasa de defecto.
    Retorna: 'Pass', 'Pending' o 'Fail'
    """
    if defect_rate <= 1.0:
        return 'Pass'
    elif defect_rate <= 5.0:
        return 'Pending'
    else:
        return 'Fail'

# Aplicamos la función a cada fila de forma vectorizada
df[inspection_col] = df[defect_col].apply(clasificar_calidad)

# ==============================================================================
# PASO 3: Generar bandera de discrepancia (¿cambió la clasificación?)
# ==============================================================================
# Esta columna es clave para el equipo de Calidad: les dice exactamente
# qué registros tenían una clasificación errónea en el sistema original.
df['Inspection_Was_Incorrect'] = df['Inspection_Original'] != df[inspection_col]

# ==============================================================================
# PASO 4: SOBREESCRITURA del dataset (Single Source of Truth)
# ==============================================================================
df.to_csv(file_path, index=False)

# ==============================================================================
# REPORTE DE RESULTADOS
# ==============================================================================
print("\n" + "="*70)
print("📊 REPORTE: CORRECCIÓN DE LÓGICA DE CALIDAD")
print("="*70)

# Distribución ANTES vs DESPUÉS
print("\n📈 Distribución de 'Inspection results':")
print("\n   ANTES (dato original del sistema):")
antes = df['Inspection_Original'].value_counts()
for estado, cantidad in antes.items():
    print(f"     • {estado:10s}: {cantidad:3d} registros ({cantidad/len(df)*100:.1f}%)")

print("\n   DESPUÉS (recalculado con umbrales objetivos):")
despues = df[inspection_col].value_counts()
for estado, cantidad in despues.items():
    print(f"     • {estado:10s}: {cantidad:3d} registros ({cantidad/len(df)*100:.1f}%)")

# Verificar si 'Fail' existe
if 'Fail' not in despues.index:
    print("\n   ⚠️  NOTA: La categoría 'Fail' tiene 0 registros.")
    print("      Motivo: Ningún SKU supera el umbral de 5.0% de defectos.")
    print("      Rango máximo actual: {:.2f}%".format(df[defect_col].max()))
    print("      Sugerencia: Considera ajustar el umbral a 3.0% o 4.0% si deseas")
    print("      segmentar los casos de mayor riesgo.")

# Registros corregidos
total_corregidos = df['Inspection_Was_Incorrect'].sum()
print(f"\n🔧 Total de registros reclasificados: {total_corregidos} de {len(df)} ({total_corregidos/len(df)*100:.1f}%)")

# Muestra de los cambios más relevantes
print("\n🔍 Muestra de registros reclasificados (primeros 5):")
cambiados = df[df['Inspection_Was_Incorrect']][
    ['SKU', defect_col, 'Inspection_Original', inspection_col]
].head(5)

# Renombrar columnas para la visualización
cambiados_display = cambiados.rename(columns={
    defect_col: 'Defect_%',
    'Inspection_Original': 'Antes',
    inspection_col: 'Ahora'
})
print(cambiados_display.to_string(index=False))

print(f"\n💾 Dataset actualizado y guardado en: '{file_path}'")
print("="*70)

🔄 Cargando dataset desde: 'supply_chain_data.csv'...
✅ Columna de defectos detectada: 'Defect_Rate_Percentage'
   Rango actual: 0.02 - 4.94

📊 REPORTE: CORRECCIÓN DE LÓGICA DE CALIDAD

📈 Distribución de 'Inspection results':

   ANTES (dato original del sistema):
     • Pending   :  41 registros (41.0%)
     • Fail      :  36 registros (36.0%)
     • Pass      :  23 registros (23.0%)

   DESPUÉS (recalculado con umbrales objetivos):
     • Pending   :  75 registros (75.0%)
     • Pass      :  25 registros (25.0%)

   ⚠️  NOTA: La categoría 'Fail' tiene 0 registros.
      Motivo: Ningún SKU supera el umbral de 5.0% de defectos.
      Rango máximo actual: 4.94%
      Sugerencia: Considera ajustar el umbral a 3.0% o 4.0% si deseas
      segmentar los casos de mayor riesgo.

🔧 Total de registros reclasificados: 67 de 100 (67.0%)

🔍 Muestra de registros reclasificados (primeros 5):
 SKU  Defect_%   Antes   Ahora
SKU0      0.23 Pending    Pass
SKU3      4.75    Fail Pending
SKU4      3.15   

In [2]:
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURACIÓN: Ruta del único dataset (Single Source of Truth)
# ==============================================================================
file_path = 'supply_chain_data.csv'

print(f"🔄 Cargando dataset desde: '{file_path}'...")
df = pd.read_csv(file_path)

# Fijamos una semilla para que los resultados sean reproducibles si se vuelve a ejecutar
np.random.seed(42)

# ==============================================================================
# PASO 1: Generar 'Order_Date' (Fecha del Pedido)
# ==============================================================================
# Usamos un rango de 2 años (2023-2024) para permitir análisis Year-over-Year (YoY) 
# y detectar estacionalidad (ej. picos de ventas en Diciembre).
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2024-12-31')
total_days = (end_date - start_date).days

random_days = np.random.randint(0, total_days + 1, size=len(df))
df['Order_Date'] = start_date + pd.to_timedelta(random_days, unit='D')

# ==============================================================================
# PASO 2: Calcular 'Manufacturing_Date' (Fecha de Fabricación)
# ==============================================================================
# Lógica de Negocio: La fabricación debe completarse ANTES de la fecha del pedido.
# Restamos al Order_Date: el tiempo de fabricación + el tiempo del proveedor + un buffer de seguridad de 1 a 5 días.
buffer_days = np.random.randint(1, 6, size=len(df))
total_production_lead_time = df['Manufacturing_Lead_Time_Days'] + df['Supplier_Lead_Time_Days'] + buffer_days

df['Manufacturing_Date'] = df['Order_Date'] - pd.to_timedelta(total_production_lead_time, unit='D')

# ==============================================================================
# PASO 3: Calcular 'Shipping_Date' (Fecha de Envío/Entrega)
# ==============================================================================
# Lógica de Negocio: El envío ocurre después del pedido, sumando los días de tránsito reales.
df['Shipping_Date'] = df['Order_Date'] + pd.to_timedelta(df['Shipping times'], unit='D')

# ==============================================================================
# PASO 4: Feature Engineering Temporal (Jerarquía de Calendario para BI)
# ==============================================================================
# Pre-calculamos estas columnas para que el dashboard sea "drag-and-drop" en Power BI/Tableau
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Quarter'] = 'Q' + df['Order_Date'].dt.quarter.astype(str)
df['Order_Month_Num'] = df['Order_Date'].dt.month
df['Order_Month_Name'] = df['Order_Date'].dt.strftime('%b') # Ej: 'Jan', 'Feb' (abreviado, ideal para ejes X)
df['Order_DayOfWeek'] = df['Order_Date'].dt.day_name()      # Ej: 'Monday', 'Tuesday'

# ==============================================================================
# PASO 5: Reorganización de Columnas para Legibilidad
# ==============================================================================
# Movemos las nuevas columnas de fecha justo después del SKU para mantener el orden lógico
base_cols = ['Product type', 'SKU']
new_date_cols = [
    'Order_Date', 'Manufacturing_Date', 'Shipping_Date', 
    'Order_Year', 'Order_Quarter', 'Order_Month_Num', 'Order_Month_Name', 'Order_DayOfWeek'
]
remaining_cols = [c for c in df.columns if c not in base_cols and c not in new_date_cols]

df = df[base_cols + new_date_cols + remaining_cols]

# ==============================================================================
# PASO 6: SOBREESCRITURA del dataset (Single Source of Truth)
# ==============================================================================
df.to_csv(file_path, index=False)

print("✅ ¡Proceso completado exitosamente!")
print(f"💾 El archivo '{file_path}' ha sido actualizado con la dimensión temporal.")

# ==============================================================================
# REPORTE DE VALIDACIÓN CRONOLÓGICA
# ==============================================================================
print("\n" + "="*70)
print("📊 VALIDACIÓN DE LÓGICA TEMPORAL")
print("="*70)

# Verificar que Manufacturing_Date <= Order_Date <= Shipping_Date
logic_check = (df['Manufacturing_Date'] <= df['Order_Date']) & (df['Order_Date'] <= df['Shipping_Date'])
print(f"• Registros con cronología lógica válida (Fabricación → Pedido → Envío): {logic_check.sum()} de {len(df)} ({(logic_check.sum()/len(df))*100:.1f}%)")

print("\n🔍 Muestra de las nuevas columnas temporales (Primeras 3 filas):")
sample_cols = ['SKU', 'Order_Date', 'Manufacturing_Date', 'Shipping_Date', 'Order_Quarter', 'Order_Month_Name']
print(df[sample_cols].head(3).to_string(index=False))

print("\n📈 Rango de fechas generado:")
print(f"   • Fecha de pedido más antigua: {df['Order_Date'].min().strftime('%Y-%m-%d')}")
print(f"   • Fecha de pedido más reciente: {df['Order_Date'].max().strftime('%Y-%m-%d')}")
print("="*70)

🔄 Cargando dataset desde: 'supply_chain_data.csv'...
✅ ¡Proceso completado exitosamente!
💾 El archivo 'supply_chain_data.csv' ha sido actualizado con la dimensión temporal.

📊 VALIDACIÓN DE LÓGICA TEMPORAL
• Registros con cronología lógica válida (Fabricación → Pedido → Envío): 100 de 100 (100.0%)

🔍 Muestra de las nuevas columnas temporales (Primeras 3 filas):
 SKU Order_Date Manufacturing_Date Shipping_Date Order_Quarter Order_Month_Name
SKU0 2023-04-13         2023-03-07    2023-04-17            Q2              Apr
SKU1 2024-03-11         2024-01-14    2024-03-13            Q1              Mar
SKU2 2023-09-28         2023-09-03    2023-09-30            Q3              Sep

📈 Rango de fechas generado:
   • Fecha de pedido más antigua: 2023-01-02
   • Fecha de pedido más reciente: 2024-12-30


In [3]:
import pandas as pd

# ==============================================================================
# CONFIGURACIÓN: Ruta del único dataset (Single Source of Truth)
# ==============================================================================
file_path = 'supply_chain_data.csv'

print(f"🔄 Cargando dataset desde: '{file_path}' para el pulido final...")
df = pd.read_csv(file_path)

# ==============================================================================
# PASO 1: Auditoría del estado actual de 'Customer demographics'
# ==============================================================================
unknown_count = (df['Customer demographics'] == 'Unknown').sum()
total_records = len(df)
percentage_unknown = (unknown_count / total_records) * 100

print(f"\n🔍 AUDITORÍA PREVIA:")
print(f"   • Registros con 'Unknown': {unknown_count} de {total_records} ({percentage_unknown:.1f}%)")

# ==============================================================================
# PASO 2: Estandarización Estética (Limpieza de Categóricas)
# ==============================================================================
# Reemplazamos 'Unknown' por 'Not_Specified' para tratarlo como categoría válida
df['Customer demographics'] = df['Customer demographics'].replace('Unknown', 'Not_Specified')

# Verificamos que no haya quedado ningún 'Unknown' residual (por si hay espacios extra)
df['Customer demographics'] = df['Customer demographics'].str.strip().replace('Unknown', 'Not_Specified')

# ==============================================================================
# PASO 3: SOBREESCRITURA del dataset (Single Source of Truth)
# ==============================================================================
df.to_csv(file_path, index=False)

print("\n✅ ¡Pulido final completado exitosamente!")
print(f"💾 El archivo '{file_path}' ha sido actualizado.")

# ==============================================================================
# REPORTE DE VALIDACIÓN FINAL
# ==============================================================================
print("\n" + "="*70)
print("📊 DISTRIBUCIÓN FINAL DE 'Customer demographics' (Lista para BI)")
print("="*70)

distribucion_final = df['Customer demographics'].value_counts()
for categoria, cantidad in distribucion_final.items():
    porcentaje = (cantidad / total_records) * 100
    print(f"   • {categoria:15s}: {cantidad:3d} registros ({porcentaje:.1f}%)")

print("\n" + "="*70)
print("🎉 ¡FELICITACIONES! DATASET AL 100% DE READINESS PARA BI")
print("="*70)
print("El dataset 'supply_chain_data.csv' ahora posee:")
print("  1. ✅ Nomenclatura estandarizada y autoexplicativa.")
print("  2. ✅ Dimensiones temporales completas (Fechas, Años, Trimestres, Meses).")
print("  3. ✅ Banderas de auditoría de calidad (Revenue, Backorders, Inspecciones).")
print("  4. ✅ Categóricas limpias y estéticas para visualización directa.")
print("\n🚀 Está listo para ser importado directamente en Power BI, Tableau, Looker o Excel.")
print("="*70)

🔄 Cargando dataset desde: 'supply_chain_data.csv' para el pulido final...

🔍 AUDITORÍA PREVIA:
   • Registros con 'Unknown': 31 de 100 (31.0%)

✅ ¡Pulido final completado exitosamente!
💾 El archivo 'supply_chain_data.csv' ha sido actualizado.

📊 DISTRIBUCIÓN FINAL DE 'Customer demographics' (Lista para BI)
   • Not_Specified  :  31 registros (31.0%)
   • Female         :  25 registros (25.0%)
   • Non-binary     :  23 registros (23.0%)
   • Male           :  21 registros (21.0%)

🎉 ¡FELICITACIONES! DATASET AL 100% DE READINESS PARA BI
El dataset 'supply_chain_data.csv' ahora posee:
  1. ✅ Nomenclatura estandarizada y autoexplicativa.
  2. ✅ Dimensiones temporales completas (Fechas, Años, Trimestres, Meses).
  3. ✅ Banderas de auditoría de calidad (Revenue, Backorders, Inspecciones).
  4. ✅ Categóricas limpias y estéticas para visualización directa.

🚀 Está listo para ser importado directamente en Power BI, Tableau, Looker o Excel.
